# 03. Model Training & Comparison

This notebook trains three different models to forecast sales: Linear Regression, Random Forest, and XGBoost. We evaluate them on a time-based train-test split using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R² score, then save the best model.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
os.makedirs("../models", exist_ok=True)

## 1. Load and Prepare Engineered Features

We will apply the feature engineering pipeline from notebook 02.

In [ ]:
# Read data
df = pd.read_csv("../data/train.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['store', 'item', 'date']).reset_index(drop=True)

# Feature engineering
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['week'] = df['date'].dt.isocalendar().week.astype(int)
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek
df['quarter'] = df['date'].dt.quarter
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

# Lags & rolling means
for lag in [7, 14, 30]:
    df[f'sales_lag_{lag}'] = df.groupby(['store', 'item'])['sales'].shift(lag)

for window in [7, 14, 30]:
    df[f'sales_roll_mean_{window}'] = df.groupby(['store', 'item'])['sales'] \
                                          .shift(1) \
                                          .transform(lambda x: x.rolling(window=window).mean())

# Drop NaNs
df = df.dropna().reset_index(drop=True)
df.head()

## 2. Time-Based Train-Test Split

Using a random train-test split for time series would cause look-ahead leakage. Instead, we split chronologically: 
- **Train**: 2022-01-01 to 2025-06-01
- **Test**: 2025-06-01 to 2025-12-31

In [ ]:
split_date = pd.to_datetime("2025-06-01")

train_df = df[df['date'] < split_date]
test_df = df[df['date'] >= split_date]

features = [
    'store', 'item', 'price',
    'year', 'month', 'week', 'day', 'dayofweek', 'quarter', 'is_weekend',
    'sales_lag_7', 'sales_lag_14', 'sales_lag_30',
    'sales_roll_mean_7', 'sales_roll_mean_14', 'sales_roll_mean_30'
]
target = 'sales'

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

## 3. Scale Features

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler
joblib.dump(scaler, "../models/scaler.pkl")

## 4. Train Models & Evaluate

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=150, learning_rate=0.08, max_depth=6, random_state=42, n_jobs=-1)
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    
    # Metric evaluations
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    }

results_df = pd.DataFrame(results).T
results_df

## 5. Identify Best Model and Save

Save the best model based on MAE/R2. Generally, XGBoost or Random Forest outperforms Linear Regression.

In [ ]:
# Find best model based on lowest MAE
best_model_name = results_df['MAE'].idxmin()
best_model = models[best_model_name]
print(f"Best model is: {best_model_name}")

# Save model (as requested: xgboost_model.pkl)
joblib.dump(best_model, "../models/xgboost_model.pkl")
print(f"Saved {best_model_name} model to ../models/xgboost_model.pkl")

## 6. Plot Feature Importance (for tree-based models)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=importances[indices], y=np.array(features)[indices], palette='viridis')
    plt.title(f"Feature Importances - {best_model_name}")
    plt.xlabel("Relative Importance")
    plt.tight_layout()
    plt.savefig("../images/feature_importance.png", dpi=150)
    plt.show()